# Prediction analysis: `class` vs `predicted_class`

Compares the trigger-word ground-truth `class` against the model's `predicted_class`
for two prediction files:

1. `filtered_events_class_with_predicted.csv` — **model** predictions.
2. `filtered_events_class_with_predicted_students.csv` — **student** predictions.

For each file we compute accuracy and inspect the mistakes. Rows whose ground-truth
`class` is `unknown` are **excluded** from the accuracy/mistake metrics — `unknown`
just means no trigger word matched, so it is not a real label to score against. The
final section compares the two files and shows what each model predicts *for* those
`unknown` events.

Paths are relative to `pipeline/`.

In [1]:
import pandas as pd
import plotly.express as px

DATA_DIR = '../../data'
MODEL_CSV = f'{DATA_DIR}/filtered_events_class_with_predicted.csv'
STUDENTS_CSV = f'{DATA_DIR}/filtered_events_class_with_predicted_students.csv'

UNKNOWN = 'unknown'

## Helpers

Shared logic so each file's section stays short. `evaluate` splits a file into the
scored rows (ground-truth `class` is not `unknown`), the subset of those that were
predicted wrong, and the overall accuracy.

In [2]:
def load(path):
    df = pd.read_csv(path, low_memory=False)
    return df[['country', 'class', 'predicted_class', 'notes', 'clean_notes']].copy()


def evaluate(df):
    """Return (scored, mistakes, accuracy) ignoring rows whose true class is unknown."""
    scored = df[df['class'] != UNKNOWN].copy()
    scored['correct'] = scored['class'] == scored['predicted_class']
    mistakes = scored[~scored['correct']]
    accuracy = scored['correct'].mean()
    return scored, mistakes, accuracy


def per_class_accuracy(scored):
    return (scored.groupby('class')['correct']
            .agg(accuracy='mean', n='size')
            .sort_values('accuracy'))


def top_confusions(mistakes, n=15):
    pairs = (mistakes.groupby(['class', 'predicted_class']).size()
             .reset_index(name='count').sort_values('count', ascending=False))
    pairs['pair'] = pairs['class'] + ' → ' + pairs['predicted_class']
    return pairs.head(n)

## 1. Model predictions — `filtered_events_class_with_predicted.csv`

In [3]:
model_df = load(MODEL_CSV)
model_scored, model_mistakes, model_acc = evaluate(model_df)

print(f'Total rows:            {len(model_df)}')
print(f'Unknown (excluded):    {(model_df["class"] == UNKNOWN).sum()}')
print(f'Scored rows:           {len(model_scored)}')
print(f'Mistakes:              {len(model_mistakes)}')
print(f'Accuracy:              {model_acc:.4f}')

Total rows:            214232
Unknown (excluded):    83921
Scored rows:           130311
Mistakes:              2621
Accuracy:              0.9799


Per-class accuracy on the scored rows, sorted worst-first. The number above each bar
is how many events of that true class were scored, so you can see whether a low bar is
a real problem or just a tiny class.

In [4]:
pca = per_class_accuracy(model_scored).reset_index()
fig = px.bar(pca, x='class', y='accuracy', color='accuracy', text='n',
             color_continuous_scale='RdYlGn', range_color=[0, 1],
             labels={'class': 'True class', 'accuracy': 'Accuracy', 'n': 'Rows'})
fig.update_layout(title={'text': 'Model: per-class accuracy', 'x': 0.5},
                  plot_bgcolor='white', width=950, height=500, yaxis={'range': [0, 1.05]})
fig.show()

The most frequent mistakes, as `true class → predicted class` pairs. These are the
specific confusions driving the error rate.

In [5]:
pairs = top_confusions(model_mistakes)
fig = px.bar(pairs[::-1], x='count', y='pair', orientation='h', text='count',
             labels={'count': 'Mistakes', 'pair': 'true → predicted'})
fig.update_layout(title={'text': 'Model: most common confusions', 'x': 0.5},
                  plot_bgcolor='white', width=950, height=500)
fig.show()

A sample of individual mistakes with the event text, so the confusions above are
concrete and easy to sanity-check.

In [6]:
model_mistakes.sample(min(10, len(model_mistakes)), random_state=0)[
    ['country', 'class', 'predicted_class', 'notes']]

,country,class,predicted_class,notes
123601,Italy,public services,immigration,"On 21 May 2023, on the anniversary of the deat..."
202511,Norway,environment,climate,"On 13 January 2026, environmental activists de..."
150145,France,labor rights,education,"On 19 March 2024, 350 civil servants, includin..."
86345,Italy,health care,women rights,"On 8 March 2022, on International Women's Day,..."
71410,Italy,labor rights,environment,"On 8 November 2021, environmental services wor..."
192245,France,labor rights,public services,"On 18 September 2025, more than 220 people, in..."
143320,France,lgbtq,farmers,"On 30 January 2024, farmers set up a road bloc..."
88165,United Kingdom,labor rights,education,"On 21 March 2022, Queen's University Belfast t..."
129789,Sweden,education,lgbtq,"On 2 September 2023, people took part in the '..."
142593,France,education,labor rights,"On 26 January 2024, with the support of SNES-F..."


## 2. Student predictions — `filtered_events_class_with_predicted_students.csv`

In [7]:
students_df = load(STUDENTS_CSV)
students_scored, students_mistakes, students_acc = evaluate(students_df)

print(f'Total rows:            {len(students_df)}')
print(f'Unknown (excluded):    {(students_df["class"] == UNKNOWN).sum()}')
print(f'Scored rows:           {len(students_scored)}')
print(f'Mistakes:              {len(students_mistakes)}')
print(f'Accuracy:              {students_acc:.4f}')

Total rows:            183080
Unknown (excluded):    72984
Scored rows:           110096
Mistakes:              1638
Accuracy:              0.9851


Per-class accuracy for the student predictions, same layout as Section 1.

In [8]:
pca = per_class_accuracy(students_scored).reset_index()
fig = px.bar(pca, x='class', y='accuracy', color='accuracy', text='n',
             color_continuous_scale='RdYlGn', range_color=[0, 1],
             labels={'class': 'True class', 'accuracy': 'Accuracy', 'n': 'Rows'})
fig.update_layout(title={'text': 'Students: per-class accuracy', 'x': 0.5},
                  plot_bgcolor='white', width=950, height=500, yaxis={'range': [0, 1.05]})
fig.show()

Most common `true → predicted` confusions for the student predictions.

In [9]:
pairs = top_confusions(students_mistakes)
fig = px.bar(pairs[::-1], x='count', y='pair', orientation='h', text='count',
             labels={'count': 'Mistakes', 'pair': 'true → predicted'})
fig.update_layout(title={'text': 'Students: most common confusions', 'x': 0.5},
                  plot_bgcolor='white', width=950, height=500)
fig.show()

A sample of individual student-prediction mistakes with the event text.

In [10]:
students_mistakes.sample(min(10, len(students_mistakes)), random_state=0)[
    ['country', 'class', 'predicted_class', 'notes']]

,country,class,predicted_class,notes
3831,Spain,women rights,palestine-israel conflict,"On 5 April 2025, at the call of Tenants' Union..."
166404,Italy,pandemic,culture,"On 31 May 2020, seasonal workers in the touris..."
143261,Italy,pandemic,policies & politics,"On 19 January 2021, a group of people dubbed t..."
169570,Greece,pandemic,women rights,"On 18 April 2020, during the night, asylum see..."
169533,United Kingdom,pandemic,environment,"On 21 April 2020, around twenty residents prot..."
125500,Germany,lgbtq,women rights,"On 12 June 2021, between 200 to 400 people, in..."
125398,Spain,pandemic,health care,"On 12 June 2021, around 200 citizens, includin..."
147429,Denmark,pandemic,labor rights,"On 27 November 2020, demonstrators, including ..."
166662,Greece,pandemic,culture,"On 29 May 2020, owners of tourist coaches prot..."
144910,France,pandemic,culture,"On 20 December 2020, around 40 persons, includ..."


## 3. Comparison

Side-by-side accuracy, then — since `unknown` rows were excluded from scoring — a look
at what each model actually predicts *for* those `unknown` events.

In [11]:
summary = pd.DataFrame({
    'source': ['Model', 'Students'],
    'accuracy': [model_acc, students_acc],
    'scored_rows': [len(model_scored), len(students_scored)],
    'mistakes': [len(model_mistakes), len(students_mistakes)],
})
display(summary)

fig = px.bar(summary, x='source', y='accuracy', color='source', text=summary['accuracy'].round(4),
             labels={'source': '', 'accuracy': 'Accuracy'})
fig.update_layout(title={'text': 'Accuracy comparison (unknown excluded)', 'x': 0.5},
                  plot_bgcolor='white', width=600, height=450,
                  yaxis={'range': [0, 1.05]}, showlegend=False)
fig.show()

,source,accuracy,scored_rows,mistakes
0,Model,0.979887,130311,2621
1,Students,0.985122,110096,1638


For every event whose ground truth is `unknown`, which real class does each source
assign it? The grouped bars show the share of `unknown` events sent to each predicted
class (as a proportion, since the two files have different numbers of `unknown` rows),
so the two models' behaviour on the unlabeled events is directly comparable.

In [12]:
def unknown_distribution(df, label):
    unk = df[df['class'] == UNKNOWN]
    dist = unk['predicted_class'].value_counts(normalize=True).rename('proportion').reset_index()
    dist.columns = ['predicted_class', 'proportion']
    dist['source'] = label
    dist['count'] = unk['predicted_class'].value_counts().values
    return dist

unk_dist = pd.concat([
    unknown_distribution(model_df, 'Model'),
    unknown_distribution(students_df, 'Students'),
], ignore_index=True)

order = (unk_dist.groupby('predicted_class')['proportion'].sum()
         .sort_values(ascending=False).index.tolist())

fig = px.bar(unk_dist, x='predicted_class', y='proportion', color='source', barmode='group',
             category_orders={'predicted_class': order}, custom_data=['count'],
             labels={'predicted_class': 'Predicted class', 'proportion': 'Share of unknown events',
                     'source': ''})
fig.update_traces(hovertemplate='%{x}<br>share=%{y:.3f}<br>count=%{customdata[0]}')
fig.update_layout(title={'text': 'Predicted classes for the unknown category', 'x': 0.5},
                  plot_bgcolor='white', width=950, height=520)
fig.show()

In [13]:
# How often do the two models predict the same label, split by unknown vs the rest?
# Join on event_id_cnty since the two files cover overlapping (not identical) events.
keep = ['event_id_cnty', 'class', 'predicted_class']
m = pd.read_csv(MODEL_CSV, usecols=keep, low_memory=False)
s = pd.read_csv(STUDENTS_CSV, usecols=keep, low_memory=False)
joined = m.merge(s, on='event_id_cnty', suffixes=('_model', '_students'))
joined['agree'] = joined['predicted_class_model'] == joined['predicted_class_students']

is_unknown = joined['class_model'] == UNKNOWN
agreement = pd.DataFrame({
    'group': ['unknown category', 'all other classes'],
    'shared_events': [is_unknown.sum(), (~is_unknown).sum()],
    'agreement': [joined.loc[is_unknown, 'agree'].mean(),
                  joined.loc[~is_unknown, 'agree'].mean()],
})
display(agreement)

fig = px.bar(agreement, x='group', y='agreement', color='group', text=agreement['agreement'].round(4),
             labels={'group': '', 'agreement': 'Model–student agreement'})
fig.update_layout(title={'text': 'How often the two models predict the same label', 'x': 0.5},
                  plot_bgcolor='white', width=600, height=450,
                  yaxis={'range': [0, 1.05]}, showlegend=False)
fig.show()

,group,shared_events,agreement
0,unknown category,72872,0.449748
1,all other classes,110065,0.976732


Every event where the two models disagree, counted as an unordered `label ↔ label`
pair across all shared events. The table lists all pairs; the chart shows the 20 most
frequent, i.e. which class confusions drive the disagreement between the two models.

In [14]:
# Overview of every model/student disagreement as an unordered label pair, to see which
# classes get mismatched between the two models most often. Reuses `joined` from above.
disagree = joined[~joined['agree']].copy()
disagree['pair'] = disagree.apply(
    lambda r: ' ↔ '.join(sorted([r['predicted_class_model'], r['predicted_class_students']])),
    axis=1)
pair_counts = disagree['pair'].value_counts().reset_index()
pair_counts.columns = ['pair', 'count']
print(f'{len(disagree)} disagreements across {len(pair_counts)} distinct label pairs')
display(pair_counts)

top = pair_counts.head(20)
fig = px.bar(top[::-1], x='count', y='pair', orientation='h', text='count',
             labels={'count': 'Disagreements', 'pair': 'model ↔ students'})
fig.update_layout(title={'text': 'Most common label mismatches between the two models', 'x': 0.5},
                  plot_bgcolor='white', width=950, height=650)
fig.show()

42659 disagreements across 179 distinct label pairs


,pair,count
0,policies & politics ↔ public services,4295
1,labor rights ↔ policies & politics,3760
2,lgbtq ↔ policies & politics,2599
3,environment ↔ policies & politics,2291
4,labor rights ↔ public services,2016
...,...,...
174,culture ↔ discrimination,1
175,blm ↔ farmers,1
176,farmers ↔ housing,1
177,climate ↔ health care,1
